# YouTube → Google Drive (Kaggle)

Downloads one video, cuts it at 1h 55m (under Dailymotion's 2-hour / 4 GB file cap), and uploads each part to a **private** named folder inside [the fixed Drive folder](https://drive.google.com/drive/folders/1Sd6KVFz8gkAjXubmt9Rx6Yjn5CSEQLIr).

Files are owned by the Google account that minted `GDRIVE_REFRESH_TOKEN`. No sharing permission is added, so they stay private and count toward that account's storage.

**One-time Drive setup**

1. In Google Cloud, enable the Drive API and create an OAuth client (Desktop).
2. On your machine: `python kaggle/gdrive_oauth.py` with `GDRIVE_CLIENT_ID` and `GDRIVE_CLIENT_SECRET` set. Approve the `drive` scope.
3. Backend env (copied into this private kernel when a job starts):
   - `API_PUBLIC_URL` (public API base ending in `/api`), `COLAB_JOB_SECRET`
   - `GDRIVE_CLIENT_ID`, `GDRIVE_CLIENT_SECRET`, `GDRIVE_REFRESH_TOKEN`
   - `YOUTUBE_COOKIES` optional if you are not attaching a cookies dataset

This kernel only claims `youtube_to_drive` jobs. When it finishes, a Drive → Dailymotion job waits until an admin presses Start.


In [ ]:
#!/usr/bin/env python3
"""Kaggle worker: download YouTube, cut to the Dailymotion limit, upload private parts to Drive.

One source video and one cut part are kept on the ~20GB Kaggle disk. Each part is
uploaded to a private named folder, then deleted locally. Nothing is shared.
"""

from __future__ import annotations

import json
import os
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

WORK_DIR = Path("/kaggle/working/ytdrive")
COOKIE_FILENAMES = ("cookies.txt", "combined_cookies.txt", "youtube_cookies.txt")
COOKIE_CANDIDATES = [
    "/kaggle/input/youtube-cookies/cookies.txt",
    "/kaggle/input/youtube-cookies/combined_cookies.txt",
    "/kaggle/input/youtube-cookies/youtube_cookies.txt",
]
# Prefer H.264 (avc1): YouTube's best "mp4" video is often AV1, which older ffmpeg
# builds fail to merge into mp4 ("Postprocessing: Conversion failed!").
QUALITY_FORMATS = {
    "Best Available": "bestvideo[vcodec^=avc1]+bestaudio[ext=m4a]/bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best",
    "4K": "bestvideo[height<=2160][vcodec^=avc1]+bestaudio[ext=m4a]/bestvideo[height<=2160]+bestaudio/best[height<=2160]/best",
    "1080p": "bestvideo[height<=1080][vcodec^=avc1]+bestaudio[ext=m4a]/bestvideo[height<=1080]+bestaudio/best[height<=1080]/best",
    "720p": "bestvideo[height<=720][vcodec^=avc1]+bestaudio[ext=m4a]/bestvideo[height<=720]+bestaudio/best[height<=720]/best",
    "480p": "bestvideo[height<=480][vcodec^=avc1]+bestaudio[ext=m4a]/bestvideo[height<=480]+bestaudio/best[height<=480]/best",
    "360p": "bestvideo[height<=360][vcodec^=avc1]+bestaudio[ext=m4a]/bestvideo[height<=360]+bestaudio/best[height<=360]/best",
}
COMBINED_FORMAT = "best[ext=mp4]/best"
YOUTUBE_DOWNLOAD_ATTEMPTS = (
    {
        "label": "default clients (no android_sdkless)",
        "extractor_args": {"youtube": {"player_client": ["default", "-android_sdkless"]}},
        "format_key": "quality",
    },
    {
        "label": "tv + mweb clients",
        "extractor_args": {"youtube": {"player_client": ["tv", "mweb", "web"]}},
        "format_key": "quality",
    },
    {
        "label": "combined progressive stream",
        "extractor_args": {"youtube": {"player_client": ["default", "-android_sdkless"]}},
        "format_key": "combined",
    },
)

def format_hms(seconds: float) -> str:
    total = max(0, int(seconds))
    hours, rem = divmod(total, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{hours}h{minutes:02d}m{secs:02d}s"
    return f"{minutes}m{secs:02d}s"

def log(message: str) -> None:
    print(message, flush=True)

def disk_status(path: Path) -> None:
    usage = shutil.disk_usage(path)
    log(f"💾 Disk: {usage.used / 1e9:.1f} GB used, {usage.free / 1e9:.1f} GB free")

def remove_path(path: str | Path | None) -> None:
    if not path:
        return
    target = Path(path)
    try:
        if target.is_dir():
            shutil.rmtree(target, ignore_errors=True)
        elif target.exists():
            target.unlink()
    except OSError as err:
        log(f"⚠️ Could not delete {target}: {err}")

def load_injected_secrets() -> dict[str, str]:
    raw = "__INJECTED_SECRETS_B64__"
    if not raw or raw.startswith("__"):
        return {}
    try:
        import base64
        import json as _json

        payload = _json.loads(base64.b64decode(raw).decode())
        if isinstance(payload, dict):
            return {str(key): str(value) for key, value in payload.items() if value}
    except Exception as err:
        log(f"Could not read injected settings: {err}")
    return {}


INJECTED_SECRETS = load_injected_secrets()


def get_secret(name: str, *aliases: str) -> str:
    keys = (name, *aliases)
    for key in keys:
        value = os.environ.get(key, "").strip()
        if value:
            return value
    try:
        from kaggle_secrets import UserSecretsClient

        client = UserSecretsClient()
        for key in keys:
            try:
                value = (client.get_secret(key) or "").strip()
            except Exception:
                value = ""
            if value:
                return value
    except Exception as err:
        log(f"Kaggle user secrets unavailable: {err}")
    for key in keys:
        value = str(INJECTED_SECRETS.get(key) or "").strip()
        if value:
            return value
    return ""

def deno_path() -> str | None:
    candidates = [
        shutil.which("deno"),
        str(Path.home() / ".deno" / "bin" / "deno"),
        "/root/.deno/bin/deno",
        "/usr/local/bin/deno",
    ]
    for candidate in candidates:
        if candidate and Path(candidate).is_file() and os.access(candidate, os.X_OK):
            return candidate
    return None

def api_headers(secret: str) -> dict[str, str]:
    return {"x-colab-secret": secret, "Content-Type": "application/json"}

def write_cookie_file(raw: str, dest: Path) -> str:
    dest.write_text(raw.replace("\\n", "\n"), encoding="utf-8")
    return str(dest)

def fetch_admin_cookies(api_base: str, secret: str) -> str | None:
    import requests

    try:
        res = requests.get(
            f"{api_base.rstrip('/')}/colab-jobs/cookies",
            headers=api_headers(secret),
            timeout=30,
        )
        if not res.ok:
            log(f"⚠️ Could not load admin cookies ({res.status_code})")
            return None
        raw = (res.json() or {}).get("cookiesTxt") or ""
        if not str(raw).strip():
            return None
        path = write_cookie_file(str(raw), Path("/tmp/admin-youtube-cookies.txt"))
        log(f"🍪 Loaded admin cookies ({len(raw.splitlines())} lines)")
        return path
    except Exception as err:
        log(f"⚠️ Could not load admin cookies: {err}")
        return None

def write_cookies_from_secret() -> str | None:
    raw = get_secret("YOUTUBE_COOKIES", "COOKIES_TXT")
    if not raw:
        return None
    return write_cookie_file(raw, Path("/tmp/youtube-cookies.txt"))

def find_cookies(job: dict) -> str | None:
    explicit = (job.get("cookieFilePath") or "").strip()
    candidates: list[str] = []
    admin_path = (job.get("_adminCookieFile") or "").strip()
    if admin_path:
        candidates.append(admin_path)
    if explicit:
        candidates.append(explicit)
    secret_path = write_cookies_from_secret()
    if secret_path:
        candidates.append(secret_path)
    candidates.extend(COOKIE_CANDIDATES)
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        for path in sorted(input_root.rglob("*")):
            if path.is_file() and path.name.lower() in COOKIE_FILENAMES:
                candidates.append(str(path))
    seen: set[str] = set()
    for path in candidates:
        if not path or path in seen:
            continue
        seen.add(path)
        if Path(path).is_file():
            return path
    return None

def prefer_mp4(path: str) -> str:
    base, _ext = os.path.splitext(path)
    if os.path.exists(base + ".mp4"):
        return base + ".mp4"
    return path

def ydl_cookie_opts(job: dict) -> dict:
    cookie_file = find_cookies(job)
    if cookie_file:
        log(f"🍪 Using cookies: {cookie_file}")
        return {"cookiefile": cookie_file}
    log(
        "⚠️ No YouTube cookies found. Upload cookies in admin, attach a Kaggle dataset "
        "(KAGGLE_COOKIE_DATASET), or add secret YOUTUBE_COOKIES — otherwise downloads often 403."
    )
    return {}

def is_youtube_forbidden(err: BaseException) -> bool:
    text = str(err).lower()
    return "403" in text or "forbidden" in text or "not a bot" in text

def is_conversion_failure(err: BaseException) -> bool:
    text = str(err).lower()
    return "postprocessing" in text or "conversion failed" in text or "ffmpeg" in text

class YdlLogger:
    """Keeps yt-dlp debug output (incl. ffmpeg stderr) quiet but available on failure."""

    def __init__(self, keep: int = 40) -> None:
        from collections import deque

        self.lines = deque(maxlen=keep)

    def debug(self, msg: str) -> None:
        self.lines.append(msg)

    def info(self, msg: str) -> None:
        self.lines.append(msg)

    def warning(self, msg: str) -> None:
        self.lines.append(f"WARNING: {msg}")

    def error(self, msg: str) -> None:
        self.lines.append(msg)

    def dump(self, title: str) -> None:
        if not self.lines:
            return
        log(f"🪵 {title} (last {len(self.lines)} yt-dlp/ffmpeg lines):")
        for line in self.lines:
            print(f"    {line}", flush=True)

def ydl_youtube_base_opts(job: dict) -> dict:
    opts: dict = {
        "noplaylist": True,
        "retries": 10,
        "fragment_retries": 10,
        "extractor_retries": 3,
        "continuedl": True,
        "concurrent_fragment_downloads": 1,
        "remote_components": ["ejs:github"],
        **ydl_cookie_opts(job),
    }
    deno = deno_path()
    if deno:
        opts["js_runtimes"] = {"deno": {"path": deno}}
    return opts

def list_video_urls(job: dict) -> list[str]:
    from yt_dlp import YoutubeDL

    youtube_url = (job.get("youtubeUrl") or "").strip()
    if not youtube_url:
        raise ValueError("Missing youtubeUrl")
    if not job.get("isPlaylist"):
        return [youtube_url]

    opts: dict = {
        "quiet": True,
        "no_warnings": True,
        "extract_flat": True,
        **ydl_youtube_base_opts(job),
        "noplaylist": False,
    }
    playlist_range = (job.get("playlistRange") or "").strip()
    if playlist_range:
        opts["playlist_items"] = playlist_range

    with YoutubeDL(opts) as ydl:
        info = ydl.extract_info(youtube_url, download=False)

    if not info or info.get("_type") != "playlist":
        return [youtube_url]

    urls: list[str] = []
    for entry in info.get("entries") or []:
        if not entry:
            continue
        url = entry.get("webpage_url") or entry.get("url") or entry.get("id")
        if not url:
            continue
        if not str(url).startswith("http"):
            url = f"https://www.youtube.com/watch?v={url}"
        urls.append(str(url))
    if not urls:
        raise RuntimeError("Playlist had no video entries")
    log(f"📃 Playlist has {len(urls)} video(s); downloading one at a time")
    return urls

def download_youtube(job: dict, work_dir: Path) -> tuple[str, str]:
    from tqdm import tqdm
    from yt_dlp import YoutubeDL

    youtube_url = (job.get("youtubeUrl") or "").strip()
    quality = "Best Available"

    work_dir.mkdir(parents=True, exist_ok=True)
    state: dict = {"bar": None}

    def progress_hook(d):
        if d["status"] == "downloading":
            total = d.get("total_bytes") or d.get("total_bytes_estimate")
            downloaded = d.get("downloaded_bytes", 0)
            if state["bar"] is None and total:
                state["bar"] = tqdm(
                    total=total,
                    unit="B",
                    unit_scale=True,
                    unit_divisor=1024,
                    desc="⬇️ Downloading",
                )
            if state["bar"] is not None:
                state["bar"].n = downloaded
                state["bar"].refresh()
        elif d["status"] == "finished" and state["bar"] is not None:
            state["bar"].close()
            state["bar"] = None
            log("🔄 Merging audio/video (if needed)...")

    base_opts = {
        "merge_output_format": "mp4",
        "outtmpl": str(work_dir / "%(title)s.%(ext)s"),
        "progress_hooks": [progress_hook],
        "quiet": True,
        # verbose + custom logger: ffmpeg's real error is only emitted as debug output
        "verbose": True,
        **ydl_youtube_base_opts(job),
    }

    disk_status(work_dir)
    last_error: Exception | None = None
    path = ""
    title = ""
    attempts = list(YOUTUBE_DOWNLOAD_ATTEMPTS)
    conversion_fallback_used = False
    index = 0
    while index < len(attempts):
        attempt = attempts[index]
        index += 1
        fmt = COMBINED_FORMAT if attempt["format_key"] == "combined" else QUALITY_FORMATS[quality]
        log(f"⬇️ Downloading {youtube_url} at {quality} ({attempt['label']})")
        ydl_logger = YdlLogger()
        ydl_opts = {
            **base_opts,
            "format": fmt,
            "extractor_args": attempt["extractor_args"],
            "logger": ydl_logger,
        }
        existing = set(work_dir.iterdir())
        try:
            with YoutubeDL(ydl_opts) as ydl:
                info = ydl.extract_info(youtube_url, download=True)
                path = prefer_mp4(ydl.prepare_filename(info))
                title = str((info or {}).get("title") or "").strip()
            last_error = None
            break
        except Exception as err:
            last_error = err
            if state["bar"] is not None:
                state["bar"].close()
                state["bar"] = None
            if index < len(attempts) and is_youtube_forbidden(err):
                log(f"⚠️ {attempt['label']} failed ({err}). Trying next YouTube client...")
                continue
            if is_conversion_failure(err) and not conversion_fallback_used:
                ydl_logger.dump("Merge failed")
                for leftover in set(work_dir.iterdir()) - existing:
                    remove_path(leftover)
                disk_status(work_dir)
                conversion_fallback_used = True
                attempts.insert(
                    index,
                    {**attempt, "label": f"{attempt['label']} → single-file fallback", "format_key": "combined"},
                )
                log("⚠️ Audio/video merge failed. Retrying with a single pre-merged mp4 stream...")
                continue
            ydl_logger.dump("Download failed")
            raise

    if last_error:
        raise last_error

    if not Path(path).exists():
        raise RuntimeError("Download finished but the video file is missing")
    if not title:
        title = Path(path).stem

    size_mb = Path(path).stat().st_size / (1024 * 1024)
    log(f"✅ Downloaded {title} ({size_mb:.1f} MB)")
    disk_status(work_dir)
    return path, title

def get_duration(filepath: str) -> float:
    cmd = ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_format", filepath]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=True)
    return float(json.loads(result.stdout)["format"]["duration"])

def extract_chunk(input_path: str, output_path: str, start: float, length: float) -> None:
    cmd = [
        "ffmpeg",
        "-y",
        "-ss",
        f"{start:.3f}",
        "-t",
        f"{length:.3f}",
        "-i",
        input_path,
        "-c",
        "copy",
        "-map",
        "0",
        "-avoid_negative_ts",
        "make_zero",
        output_path,
    ]
    result = subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg split failed: {result.stderr[-500:]}")

DM_MAX_FILE_BYTES = 4 * 1024 * 1024 * 1024
DM_MAX_FILE_SECONDS = 2 * 3600
JOB_KIND = "youtube_to_drive"


class JobCheckpoint:
    def __init__(
        self,
        next_part: int,
        resume_at_seconds: float,
        drive_files: list[dict],
        folder_id: str | None,
        folder_name: str | None,
        folder_url: str | None,
    ) -> None:
        self.urls: list[dict] = []
        self.next_part = next_part
        self.resume_at_seconds = float(resume_at_seconds)
        self.drive_files = list(drive_files)
        self.folder_id = folder_id
        self.folder_name = folder_name
        self.folder_url = folder_url

    @classmethod
    def from_job(cls, job: dict) -> "JobCheckpoint":
        return cls(
            next_part=int(job.get("startPartNumber") or 1),
            resume_at_seconds=float(job.get("resumeAtSeconds") or 0),
            drive_files=list(job.get("driveFiles") or []),
            folder_id=(job.get("driveFolderId") or None),
            folder_name=(job.get("driveFolderName") or None),
            folder_url=(job.get("driveFolderUrl") or None),
        )

    def payload(self) -> dict:
        body: dict = {
            "urls": self.urls,
            "nextStartPart": self.next_part,
            "resumeAtSeconds": int(self.resume_at_seconds),
            "driveFiles": self.drive_files,
        }
        if self.folder_id:
            body["driveFolderId"] = self.folder_id
        if self.folder_name:
            body["driveFolderName"] = self.folder_name
        if self.folder_url:
            body["driveFolderUrl"] = self.folder_url
        return body


class PipelineStop(Exception):
    def __init__(self, message: str, checkpoint: JobCheckpoint, *, paused: bool) -> None:
        super().__init__(message)
        self.checkpoint = checkpoint
        self.paused = paused


def ensure_tools() -> None:
    log("📦 Installing yt-dlp, Google Drive client, tqdm...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U", "yt-dlp[default]", "tqdm"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "google-api-python-client"],
        check=True,
    )
    deno_bin = str(Path.home() / ".deno" / "bin")
    if deno_bin not in os.environ.get("PATH", ""):
        os.environ["PATH"] = deno_bin + os.pathsep + os.environ.get("PATH", "")
    if deno_path() is None:
        log("📦 Installing Deno (yt-dlp YouTube JS challenges)...")
        subprocess.run(
            ["bash", "-c", "curl -fsSL https://deno.land/install.sh | sh -s -- -y"],
            check=True,
        )
    try:
        import yt_dlp

        log(f"yt-dlp {yt_dlp.version.__version__}")
    except Exception:
        pass
    deno = deno_path()
    if deno:
        log(f"🦕 Deno JS runtime: {deno}")
    else:
        log("⚠️ Deno not on PATH — YouTube JS challenges will fail")


def claim_job(api_base: str, secret: str) -> dict | None:
    import requests

    res = requests.get(
        f"{api_base.rstrip('/')}/colab-jobs/pending",
        headers=api_headers(secret),
        params={"kind": JOB_KIND},
        timeout=30,
    )
    res.raise_for_status()
    return res.json().get("job")


def post_checkpoint(api_base: str, secret: str, job_id: int, path: str, checkpoint: JobCheckpoint) -> None:
    import requests

    res = requests.post(
        f"{api_base.rstrip('/')}/colab-jobs/{job_id}/{path}",
        headers=api_headers(secret),
        json=checkpoint.payload(),
        timeout=60,
    )
    res.raise_for_status()


def save_progress(api_base: str, secret: str, job_id: int, checkpoint: JobCheckpoint) -> None:
    try:
        post_checkpoint(api_base, secret, job_id, "progress", checkpoint)
        log(
            f"💾 Checkpoint: {len(checkpoint.drive_files)} Drive file(s), next part {checkpoint.next_part} "
            f"at {format_hms(checkpoint.resume_at_seconds)}"
        )
    except Exception as err:
        log(f"⚠️ Could not save progress to backend: {err}")


def complete_job(api_base: str, secret: str, job_id: int, checkpoint: JobCheckpoint) -> None:
    post_checkpoint(api_base, secret, job_id, "complete", checkpoint)


def fail_job(
    api_base: str,
    secret: str,
    job_id: int,
    error_message: str,
    *,
    paused: bool = False,
    checkpoint: JobCheckpoint | None = None,
) -> None:
    import requests

    payload: dict = {"errorMessage": error_message[:4000], "paused": paused}
    if checkpoint is not None:
        payload.update(checkpoint.payload())
    try:
        requests.post(
            f"{api_base.rstrip('/')}/colab-jobs/{job_id}/fail",
            headers=api_headers(secret),
            json=payload,
            timeout=60,
        ).raise_for_status()
    except Exception as err:
        log(f"⚠️ Could not report failure to backend: {err}")


def raise_pause(reason: str, checkpoint: JobCheckpoint) -> None:
    message = (
        f"{reason} Saved {len(checkpoint.drive_files)} Drive file(s). "
        f"Resume at part {checkpoint.next_part} ({format_hms(checkpoint.resume_at_seconds)})."
    )
    raise PipelineStop(message, checkpoint, paused=True)


def sanitize_name(name: str) -> str:
    cleaned = re.sub(r'[\\/:*?"<>|]+', " ", name).strip()
    cleaned = re.sub(r"\s+", " ", cleaned)
    return (cleaned or "Untitled")[:180]


def escape_drive_query(value: str) -> str:
    return value.replace("\\", "\\\\").replace("'", "\\'")


def folder_url(folder_id: str) -> str:
    return f"https://drive.google.com/drive/folders/{folder_id}"


FIXED_DRIVE_PARENT_FOLDER_ID = "1Sd6KVFz8gkAjXubmt9Rx6Yjn5CSEQLIr"


def parent_folder_id(job: dict) -> str:
    explicit = (job.get("driveParentFolderId") or "").strip()
    if explicit:
        return explicit
    secret = get_secret("GDRIVE_PARENT_FOLDER").strip()
    match = re.search(r"/folders/([a-zA-Z0-9_-]+)", secret)
    if match:
        return match.group(1)
    if re.fullmatch(r"[a-zA-Z0-9_-]{10,}", secret):
        return secret
    return FIXED_DRIVE_PARENT_FOLDER_ID


def drive_service(parent_id: str):
    from google.oauth2.credentials import Credentials
    from googleapiclient.discovery import build

    client_id = get_secret("GDRIVE_CLIENT_ID")
    client_secret = get_secret("GDRIVE_CLIENT_SECRET")
    refresh_token = get_secret("GDRIVE_REFRESH_TOKEN")
    if not client_id or not client_secret or not refresh_token:
        raise RuntimeError(
            "Missing GDRIVE_CLIENT_ID, GDRIVE_CLIENT_SECRET, or GDRIVE_REFRESH_TOKEN. "
            "Run kaggle/gdrive_oauth.py once and store the refresh token as a Kaggle secret."
        )
    creds = Credentials(
        token=None,
        refresh_token=refresh_token,
        client_id=client_id,
        client_secret=client_secret,
        token_uri="https://oauth2.googleapis.com/token",
        scopes=["https://www.googleapis.com/auth/drive"],
    )
    service = build("drive", "v3", credentials=creds, cache_discovery=False)
    try:
        service.files().get(fileId=parent_id, fields="id,name,mimeType", supportsAllDrives=True).execute()
    except Exception as err:
        raise RuntimeError(
            "Google Drive rejected the parent folder. Check the folder link and the refresh token. "
            f"{err}"
        ) from err
    return service


def ensure_named_folder(service, parent_id: str, name: str) -> dict[str, str]:
    safe = sanitize_name(name)
    query = (
        "mimeType = 'application/vnd.google-apps.folder' and "
        f"name = '{escape_drive_query(safe)}' and "
        f"'{parent_id}' in parents and trashed = false"
    )
    found = (
        service.files()
        .list(
            q=query,
            fields="files(id,name,webViewLink)",
            pageSize=5,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        )
        .execute()
    )
    files = found.get("files") or []
    if files:
        item = files[0]
        log(f"📁 Reusing private Drive folder {item['name']}")
        return {
            "id": item["id"],
            "name": item["name"],
            "url": item.get("webViewLink") or folder_url(item["id"]),
        }
    created = (
        service.files()
        .create(
            body={
                "name": safe,
                "mimeType": "application/vnd.google-apps.folder",
                "parents": [parent_id],
            },
            fields="id,name,webViewLink",
            supportsAllDrives=True,
        )
        .execute()
    )
    log(f"📁 Created private Drive folder {created['name']} (not shared)")
    return {
        "id": created["id"],
        "name": created["name"],
        "url": created.get("webViewLink") or folder_url(created["id"]),
    }


def upload_private_file(service, folder_id: str, path: str, name: str) -> dict:
    from googleapiclient.http import MediaFileUpload

    media = MediaFileUpload(path, mimetype="video/mp4", resumable=True, chunksize=8 * 1024 * 1024)
    request = service.files().create(
        body={"name": name, "parents": [folder_id]},
        media_body=media,
        fields="id,name,size",
        supportsAllDrives=True,
    )
    response = None
    while response is None:
        status, response = request.next_chunk()
        if status:
            log(f"   ☁️ Drive upload {int(status.progress() * 100)}%")
    log(f"   🔒 Uploaded private file {response.get('name')} (no sharing permission added)")
    return response


def remember_folder(checkpoint: JobCheckpoint, folder: dict[str, str]) -> None:
    checkpoint.folder_id = folder["id"]
    checkpoint.folder_name = folder["name"]
    checkpoint.folder_url = folder["url"]


def split_to_drive(
    video_path: str,
    source_url: str,
    service,
    folder: dict[str, str],
    job: dict,
    checkpoint: JobCheckpoint,
    on_progress,
) -> None:
    chunk_seconds = int(job.get("splitHours") or 1) * 3600 + int(job.get("splitMinutes") or 55) * 60
    if chunk_seconds <= 0 or chunk_seconds > DM_MAX_FILE_SECONDS:
        raise_pause("Split length must be between 1 second and Dailymotion's 2-hour per-file limit.", checkpoint)
    title = (job.get("_videoTitle") or "").strip() or Path(video_path).stem
    duration = get_duration(video_path)
    known = {int(item.get("partNumber") or 0): item for item in checkpoint.drive_files}
    start = checkpoint.resume_at_seconds
    part = checkpoint.next_part
    if start <= 0 and part > 1 and part not in known:
        start = max(part - 1, 0) * float(chunk_seconds)

    while start < duration - 0.5:
        length = min(float(chunk_seconds), duration - start)
        existing = known.get(part)
        if existing:
            log(f"⏭️ Part {part} is already on Drive")
            start += float(existing.get("durationSeconds") or length)
            part += 1
            checkpoint.next_part = part
            checkpoint.resume_at_seconds = start
            continue
        if length > DM_MAX_FILE_SECONDS + 1:
            raise_pause(
                f"Part {part} would be {format_hms(length)}, over Dailymotion's 2-hour per-file limit.",
                checkpoint,
            )
        part_path = str(Path(video_path).with_name(f"_upload_part_{part:03d}.mp4"))
        file_name = f"{sanitize_name(title)} - Part {part}.mp4"
        log(f"✂️ Extracting part {part} ({format_hms(start)} → {format_hms(start + length)})")
        try:
            extract_chunk(video_path, part_path, start, length)
            size = Path(part_path).stat().st_size
            if size > DM_MAX_FILE_BYTES:
                raise_pause(
                    f"Part {part} is {size / (1024 ** 3):.2f} GB, over Dailymotion's 4 GB per-file limit. "
                    "Lower the quality and queue a new job.",
                    checkpoint,
                )
            uploaded = upload_private_file(service, folder["id"], part_path, file_name)
            record = {
                "fileId": uploaded["id"],
                "name": uploaded.get("name") or file_name,
                "partNumber": part,
                "bytes": int(uploaded.get("size") or size),
                "durationSeconds": int(round(length)),
                "sourceUrl": source_url,
            }
            checkpoint.drive_files.append(record)
            known[part] = record
            start += length
            part += 1
            checkpoint.next_part = part
            checkpoint.resume_at_seconds = start
            if on_progress:
                on_progress(checkpoint)
        finally:
            remove_path(part_path)
            disk_status(WORK_DIR)


def run_pipeline(job: dict, service, on_progress=None) -> JobCheckpoint:
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    checkpoint = JobCheckpoint.from_job(job)
    parent_id = parent_folder_id(job)
    urls = list_video_urls(job)
    resume_url = ""
    if checkpoint.drive_files:
        resume_url = str(checkpoint.drive_files[-1].get("sourceUrl") or "")
    skipping = bool(resume_url)

    try:
        for index, video_url in enumerate(urls, start=1):
            if skipping and video_url != resume_url:
                log(f"⏭️ Video {index} already stored on Drive")
                continue
            if skipping and video_url == resume_url:
                skipping = False
            elif checkpoint.drive_files and video_url != resume_url:
                checkpoint.resume_at_seconds = 0
            log(f"\n===== Video {index}: {video_url} =====")
            single = {**job, "youtubeUrl": video_url, "isPlaylist": False, "customTitle": None}
            downloaded, video_title = download_youtube(single, WORK_DIR)
            single["_videoTitle"] = video_title
            try:
                folder = ensure_named_folder(service, parent_id, video_title)
                remember_folder(checkpoint, folder)
                if on_progress:
                    on_progress(checkpoint)
                split_to_drive(downloaded, video_url, service, folder, single, checkpoint, on_progress)
                checkpoint.resume_at_seconds = 0
            finally:
                remove_path(downloaded)
                disk_status(WORK_DIR)
        if not checkpoint.drive_files:
            raise RuntimeError("Pipeline finished with no Drive files")
        return checkpoint
    except PipelineStop:
        raise
    except Exception as err:
        raise PipelineStop(str(err), checkpoint, paused=False) from err


def main() -> None:
    log("🧰 Worker build: youtube-to-gdrive")
    ensure_tools()
    api_base = get_secret("API_BASE")
    secret = get_secret("COLAB_JOB_SECRET", "JOB_SECRET")
    if not api_base or not secret:
        missing = []
        if not api_base:
            missing.append("API_BASE")
        if not secret:
            missing.append("COLAB_JOB_SECRET")
        raise SystemExit(
            "Missing "
            + " and ".join(missing)
            + ". Set API_PUBLIC_URL and COLAB_JOB_SECRET on the backend, then queue the job again."
        )

    processed = 0
    while True:
        job = claim_job(api_base, secret)
        if job is None:
            break
        job_id = job.get("id")
        log(f"\n{'=' * 60}\n▶ Drive job {job_id} — {job.get('youtubeUrl')}\n{'=' * 60}")
        admin_cookies = fetch_admin_cookies(api_base, secret)
        if admin_cookies:
            job["_adminCookieFile"] = admin_cookies
        try:
            service = drive_service(parent_folder_id(job))

            def on_progress(checkpoint: JobCheckpoint) -> None:
                if job_id is not None:
                    save_progress(api_base, secret, job_id, checkpoint)

            checkpoint = run_pipeline(job, service, on_progress)
            if job_id is None:
                raise RuntimeError("Claimed job had no id")
            complete_job(api_base, secret, job_id, checkpoint)
            log(
                f"✅ Job #{job_id} stored {len(checkpoint.drive_files)} private part(s) in "
                f"{checkpoint.folder_name}. A Dailymotion job is waiting for Start."
            )
        except PipelineStop as stop:
            kind = "paused" if stop.paused else "failed"
            log(f"{'⏸️' if stop.paused else '❌'} Job {job_id} {kind}: {stop}")
            if job_id is not None:
                fail_job(
                    api_base,
                    secret,
                    job_id,
                    str(stop),
                    paused=stop.paused,
                    checkpoint=stop.checkpoint,
                )
        except Exception as err:
            log(f"❌ Job {job_id} failed: {err}")
            if job_id is not None:
                fail_job(api_base, secret, job_id, str(err), checkpoint=JobCheckpoint.from_job(job))
        processed += 1
        remove_path(WORK_DIR)

    remove_path(WORK_DIR)
    if processed == 0:
        log("No pending YouTube → Drive jobs")
    else:
        log(f"Done. Processed {processed} Drive job(s).")


if __name__ == "__main__":
    main()
